# Bài 22 — chuỗi luyện Easy · LeetCode #112 — Path Sum

**Mức độ:** Easy · **Chủ đề:** Tree DFS · Target Reduction · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/path-sum/](https://leetcode.com/problems/path-sum/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


**Lộ trình đầy đủ:** đề → nền tảng → baseline → điểm mấu chốt → code → trace → chứng minh → nộp LeetCode → Big-O → lỗi → biến thể → 6 bài tập → gợi ý → đáp án → checklist.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho gốc cây và `targetSum`. Kiểm tra có đường **gốc → lá** nào có tổng giá trị node bằng `targetSum` không. Không dừng sớm ở node giữa dù đã đủ tổng.

### Ví dụ quan trọng

Cây `[5,4,8,11,None,13,4,7,2,None,None,None,1]`, `targetSum=22 → True` qua `5→4→11→2`. Cây rỗng → False.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. Từ một nút đến cây nhị phân (Binary Tree)

*TreeNode* là một **đối tượng** có 3 thuộc tính: `.val` giữ số, `.left` tham chiếu con trái, `.right` tham chiếu con phải.
Hai con có thể là `None` nghĩa là nút con không tồn tại. Nút không có cả hai con gọi là **lá** (*leaf*).
Một đường gốc → lá là chuỗi các nút liên tiếp nối bằng con trỏ; không được nhảy qua nút.

Ví dụ `root=TreeNode(5)` rồi gắn hai con 4 và 8: gốc ở **độ sâu 1**, hai con ở **độ sâu 2**. 
Đếm độ sâu theo **số nút** trên đường, không theo số cạnh. Cây rỗng có độ sâu 0.
Trước khi chạy, dự đoán giá trị của 5 phép đọc thuộc tính sau.

In [ ]:
simple_root = TreeNode(5)
simple_root.left = TreeNode(4)
simple_root.right = TreeNode(8)
print(simple_root.val, simple_root.left.val, simple_root.right.val)
print('con trái của 4:',simple_root.left.left)
print('gốc có là lá?',simple_root.left is None and simple_root.right is None)
print('nút 4 có là lá?', simple_root.left.left is None and simple_root.left.right is None)

### 2.2. Vì sao phải kiểm tra `None` trước khi truy cập `.val`?

`None` nghĩa là không có đối tượng. Viết `node.val` khi `node is None` sẽ gây `AttributeError`.
Vì thế thuật toán cây thường mở bằng `if node is None: ...` để làm **trường hợp dừng**.
`is None` kiểm tra chính xác giá trị đặc biệt `None`; `if node:` kiểm tra tính chân trị và đôi khi gây nhầm khi
đối tượng có định nghĩa quy tắc `bool` riêng.

Dấu `and` và `or` có **short-circuit**: Python dừng đánh giá phần sau khi đã biết kết quả. 
Ví dụ `node is not None and node.val > 0` an toàn: nếu node rỗng thì không thử đọc `.val`.
Còn `node.val > 0 and node is not None` không an toàn vì đổi thứ tự!

In [ ]:
missing_demo = None
present_demo = TreeNode(0)
for candidate in (missing_demo,present_demo,TreeNode(-3),TreeNode(7)):
    ok = candidate is not None and candidate.val > 0
    print('ứng viên:',candidate,'-> giá trị dương?',ok)
assert missing_demo is None

### 2.3. Tham chiếu và sự khác biệt giữa giá trị / vị trí nút

`a = root.left` không tạo ra một bản sao: `a` chỉ là tên khác tham chiếu cùng đối tượng.
Trong cây, **vị trí** một nút quan trọng hơn việc một số xuất hiện bao nhiêu lần; hai nút có cùng `.val`
vẫn là hai đối tượng khác nhau. `is` hỏi có phải cùng đối tượng; `==` hỏi hai giá trị bằng nhau.

Khi giải bài đường đi, ta cần đi theo `left/right` thực sự, không được lấy các số từ mảng mô tả cây rồi
cộng/so theo thứ tự phần tử. Dữ liệu `[1,2,None,3]` biểu diễn cây khác `[1,None,2,3]`.

In [ ]:
a_node = TreeNode(9)
b_node = TreeNode(9)
alias = a_node
print('hai giá trị:', a_node.val == b_node.val)
print('có cùng đối tượng:',a_node is b_node)
print('alias có cùng đối tượng:',alias is a_node)
print('hai cấu trúc khác:', show_tree(tree([1,2,None,3])),show_tree(tree([1,None,2,3])))

### 2.4. Danh sách level-order khác với cây trong bộ nhớ

LeetCode viết cây dưới dạng danh sách *level-order* (từng tầng, trái đến phải), chèn `null`
khi một nhánh không tồn tại. Ở Python notebook này dùng `None` thay chữ `null` trong đề.
Hàm `tree(values)` chỉ là công cụ kiểm thử: lần lượt lấy **nút cha** khỏi queue rồi gắn hai nút con.

Khi nộp LeetCode, tham số `root` **đã là TreeNode**, không phải `list`.
Vì vậy lời giải không cần `tree(...)`, cũng không được truy cập `root[0]`.
`show_tree` là hàm phụ giúp nhìn cấu trúc; không cần nộp nó.

In [ ]:
shape1=tree([1,2,3,None,4])
print('gốc',shape1,'trái',shape1.left,'phải',shape1.right)
print('con phải của 2',shape1.left.right)
print('đọc lại bằng level-order:',show_tree(shape1))
print('cây rỗng:',tree([]),show_tree(None))

### 2.5. Đệ quy từ đầu: trường hợp dừng, lời gọi con, kết quả trả về

**Recursion (đệ quy)** là hàm gọi lại chính nó nhưng trên đầu vào nhỏ hơn.
Hai điều kiện để không chạy vô hạn: (1) có **base case**; (2) mỗi lời gọi con tiến gần base case.
Với cây: `None` hoặc lá là trường hợp nhỏ nhất; `node.left` / `node.right` là cây con nhỏ hơn.

**Call stack** giữ các lời gọi đang chờ `return`. Nếu cây có chiều cao `h`, đệ quy DFS có thể cần `O(h)`
bộ nhớ ngăn xếp. Trong Python, cây quá lệch có thể chạm giới hạn đệ quy;
BFS dùng queue tránh vấn đề ấy, còn DFS lặp dùng stack tường minh.

In [ ]:
def countdown_foundation(n,indent=0):
    print('  '*indent+'vào',n)
    if n == 0:
        return 0
    child = countdown_foundation(n-1,indent+1)
    result = child + 1
    print('  '*indent+'quay lại',n,'kết quả',result)
    return result
assert countdown_foundation(3)==3

### 2.6. `return` khác `print`: LeetCode cần giá trị đúng

`print(x)` chỉ hiển thị x trên màn hình; lệnh không trả x từ hàm.
`return x` vừa kết thúc lời gọi hiện tại, vừa gửi x về nơi gọi.
Type hint như `Optional[TreeNode]` nói tham số có thể là `TreeNode` hoặc `None`;
nó giúp đọc code nhưng không thay phần kiểm tra thực tế.

Một hàm trả `bool` phải thực sự `return True/False`; hàm trả `int` phải trả con số đúng.
Đừng đặt mọi phép `print` vào hàm nộp LeetCode: chúng chỉ phục vụ quan sát trong notebook.

In [ ]:
def only_print_demo(x):
    print('bên trong:',x)
def really_return_demo(x):
    return x
print('kết quả chỉ print:', only_print_demo(42))
print('kết quả return:', really_return_demo(42))
assert only_print_demo(0) is None
assert really_return_demo(0) == 0

### 2.7. Tầng sâu, số nút và cách tạo cây phản ví dụ

Ký hiệu `n` là tổng nút thật, `h` là độ cao lớn nhất, `w` là số nút đông nhất trong một tầng.
`n`, `h`, `w` không phải cùng một đại lượng.

- Cây hình dây có `h=n`, mỗi tầng tối đa một nút.
- Cây rất đầy có `h` nhỏ (cỡ log₂ n), nhưng tầng cuối có thể chứa nhiều nút.
- Khi thuật toán cần **ngắn nhất đến lá**, một nhánh `None` **không phải** là một lá.
- Khi bài hỏi **tổng đường đi**, tổng của node ở giữa đường không đủ nếu chưa chạm lá.

Hãy dùng hai cây sau để dự đoán độ sâu tối đa và vị trí lá; đây là cách dựng phản ví dụ có chủ đích.

In [ ]:
chain_demo = tree([1,None,2,None,3,None,4])
fork_demo = tree([1,2,3,4,None,None,5])
for root_demo in (chain_demo,fork_demo):
    q_demo=deque([(root_demo,1)])
    leaf_info=[]
    while q_demo:
        node,depth=q_demo.popleft()
        if node.left is None and node.right is None:leaf_info.append((node.val,depth))
        if node.left:q_demo.append((node.left,depth+1))
        if node.right:q_demo.append((node.right,depth+1))
    print('cây:',show_tree(root_demo),'-> các lá và độ sâu:',leaf_info)

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Phép trừ "phần còn lại": nếu tổng cần là 22 và đã đi qua node 5, nhánh con còn phải góp `22-5=17`. Node **lá** có `left=None` **và** `right=None`.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
target=22;values=[5,4,11,2]
remaining=target
for x in values:
    remaining-=x
    print('vừa đi qua',x,'-> còn thiếu',remaining)
print('khớp?',remaining==0)

### 3.1. Tổng tích lũy và “số còn thiếu” là hai cách nhìn tương đương

Giả sử mục tiêu là 22, ta bắt đầu ở nút gốc 5. Nếu giữ **tổng đã đi** `total`, ban đầu `0` và
sau khi gặp 5, total thành 5. Nếu giữ **mục tiêu còn thiếu** `remaining`, ban đầu `22` và sau 5 thành 17.
Dù chọn cách nào, ta đều có bất biến `total + remaining = target`.

Qua đường `5 → 4 → 11 → 2`, các giá trị còn thiếu lần lượt là `17,13,2,0`.
Chỉ khi **đã đến một nút lá** và giá trị còn thiếu thành `0` mới kết luận tìm được đường hợp lệ.
Nếu đường có số âm, remaining có thể tăng lại, không được vội bỏ nhánh khi remaining âm.

In [ ]:
target112=22;total112=0;remaining112=target112
for value in (5,4,11,2):
    before=(total112,remaining112)
    total112 += value; remaining112 -= value
    print(f'gặp {value}: trước {before} -> đã đi {total112}, còn thiếu {remaining112}')
    assert total112+remaining112==target112
assert remaining112==0

### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

DFS mang theo tổng từ gốc. Ở node lá, so tổng đã tích lũy với đích. Cách này rõ ràng nhưng mỗi bước phải giữ tổng đã đi qua.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(root,target):
    def walk(node,total):
        if node is None:return False
        total+=node.val
        if node.left is None and node.right is None:return total==target
        return walk(node.left,total) or walk(node.right,total)
    return walk(root,0)

In [ ]:
r=tree([5,4,8,11,None,13,4,7,2,None,None,None,1]);print(baseline(r,22),baseline(r,5))

### 4.1. Vì sao phải đi từ gốc và dừng đúng tại lá?

Một **path** trong đề #112 là đường bắt đầu ở root và kết thúc tại một **leaf**.
Không phải đoạn bất kỳ nằm giữa hai node và cũng không phải tổng của mọi nút trong cây.
Ví dụ cây `[1,2,3]` và target 1: gốc có giá trị 1 nhưng gốc không phải lá, nên đáp án **False**.
Ví dụ target 3: đường `1→2` có tổng 3 và nút 2 là lá, đáp án **True**.

Một lời giải chỉ viết `if total == target: return True` ngay ở mỗi node sẽ sai.
Đặc biệt khi có node bằng 0, một tổng đúng sớm có thể đi qua các con rồi thay đổi,
không thể xem “trúng target” ở node giữa là hoàn thành đường đi.

In [ ]:
example112=tree([1,2,3])
print('chỉ gốc bằng target 1 nhưng không phải lá:',baseline(example112,1))
print('đường 1 -> 2 bằng target 3:',baseline(example112,3))
print('đường 1 -> 3 bằng target 4:',baseline(example112,4))
assert baseline(example112,1) is False
assert baseline(example112,3) is True

### 4.2. Cách baseline đưa `total` vào đệ quy như thế nào?

Trong `baseline`, hàm con `walk(node,total)` nhận **tổng của các nút cha** đã đi qua.
Khi vào node mới: cộng `node.val` vào total. Nếu đang ở lá, so sánh total với target.
Nếu chưa là lá, gọi đệ quy hai nhánh với tổng mới; `or` nghĩa là chỉ cần **một** nhánh hợp lệ.

Ta cũng có thể sửa `walk` để in độ sâu lời gọi mà không đổi logic:
`indent='  '*depth` giúp nhìn cây lời gọi.
Lưu ý Python xử lý `left or right` theo **short-circuit**: gặp True bên trái thì không cần duyệt phải.
Vì vậy thời gian có thể dừng sớm, nhưng tệ nhất vẫn phải xét n node.

In [ ]:
def trace_accumulated_total(node,target,total=0,depth=0):
    if node is None:
        print('  '*depth+'None -> False')
        return False
    total+=node.val
    print('  '*depth+f'node {node.val} tổng={total}')
    if node.left is None and node.right is None:
        return total==target
    return trace_accumulated_total(node.left,target,total,depth+1) or trace_accumulated_total(node.right,target,total,depth+1)
assert trace_accumulated_total(tree([1,2,3]),3)

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Thay vì mang `total` đã qua, mang `remaining` còn thiếu; đi vào node giá trị `x` thì cập nhật `remaining-x`. Ở lá, chỉ cần kiểm tra `remaining == node.val`.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Đệ quy nhận `(node,remaining)`. Nếu node None trả False; nếu node lá so `node.val == remaining`; còn lại gọi trái/phải với `remaining-node.val`.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Chuyển công thức tổng đã đi sang mục tiêu còn thiếu

Hai hướng biểu diễn cho cùng bài toán:

- `walk(node,total)`: cộng `node.val` rồi hỏi `total == target` khi đến lá.
- `solve(node,remaining)`: trừ `node.val` rồi hỏi `remaining == node.val` khi đến lá.

Cách hai thường đơn giản hơn vì **không cần giữ thêm biến target gốc** trong lời gọi con.
Đây không phải tối ưu Big-O: cả hai vẫn O(n) time tệ nhất, O(h) stack.
Bước đột phá ở đây là làm cho trạng thái đệ quy ngắn và dễ giải thích:
“Cây con này có đường tới lá với tổng **còn cần** bằng bao nhiêu?”

In [ ]:
root_small112=tree([4,3,9])
for target in (7,13,4):
    rem_after_root=target-root_small112.val
    possible_leaf_values=(root_small112.left.val,root_small112.right.val)
    print('target',target,'sau gốc còn',rem_after_root,'hai lá:',possible_leaf_values)
    assert baseline(root_small112,target)==(rem_after_root in possible_leaf_values)

In [ ]:
def solve(root: Optional[TreeNode], targetSum: int) -> bool:
    def walk(node,remaining):
        if node is None:return False
        if node.left is None and node.right is None:
            return node.val==remaining
        rest=remaining-node.val
        return walk(node.left,rest) or walk(node.right,rest)
    return walk(root,targetSum)

### 6.1. Giải thích chính xác ba trường hợp dừng của lời giải chính

**Trường hợp 1: `root is None`.** Không có nút thật, không thể có đường gốc→lá, nên trả `False`.
**Trường hợp 2: root là lá.** Đường chỉ còn một nút, trả `root.val == targetSum`.
**Trường hợp 3: root còn con.** Trừ giá trị root khỏi mục tiêu, rồi đặt câu hỏi tương tự cho các cây con.

Ta không được chuyển ngay sang `None` và xét `(targetSum == 0)`;
`None` là khoảng trống, không phải lá thật. Các quy tắc này đúng với số 0 và số âm.
Khi đọc code, hãy tự chỉ ra điều kiện nào ngăn truy cập `root.left` khi root rỗng.

In [ ]:
for root_example,target_example in [(None,0),(TreeNode(0),0),(TreeNode(-4),-4),(TreeNode(-4),0)]:
    print('root',root_example,'target',target_example,'=>',solve(root_example,target_example))
assert not solve(None,0)
assert solve(TreeNode(0),0)
assert solve(TreeNode(-4),-4)

### 6.2. `or` có thể bỏ qua nhánh phải — đúng hay sai?

Khi bài hỏi *có tồn tại ít nhất một đường hợp lệ không*, phép `or` là phép gộp chính xác:
`True or False` và `True or True` đều là `True`; chỉ hai bên đều False mới trả False.
Python còn tối ưu: nếu nhánh trái trả True, `or` **không gọi** nhánh phải.
Điều đó không phải là bỏ sót đáp án, vì yêu cầu chỉ trả một boolean.

Nhưng khi biến thể đổi thành **đếm tất cả đường**, thay `or` bằng `+` cho kết quả số lượng;
không thể dừng sớm ở nhánh trái vì sẽ đếm thiếu. Đây là ví dụ quan trọng về việc
**kiểu đáp án quyết định cách gộp kết quả các cây con**.

In [ ]:
call_counter112={'left':0,'right':0}
def left_true():
    call_counter112['left']+=1
    return True
def right_false():
    call_counter112['right']+=1
    return False
result_or=left_true() or right_false()
print('kết quả:',result_or,'số lần gọi:',call_counter112)
assert result_or and call_counter112['right']==0

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Node đang đến | remaining trước | remaining cho con | Ghi chú |
|---|---:|---:|---|
| `5` | 22 | 17 | còn phải được 17 |
| `4` | 17 | 13 | đi nhánh trái |
| `11` | 13 | 2 | thử con |
| `2` (lá) | 2 | — | **2==2 → True** |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

### 7.1. Trace tay bảng remaining trên nhánh thành công

Với cây ví dụ #112 và target=22, một đường thành công là `5 → 4 → 11 → 2`.

| Tầng | Nút đang xét | Mục tiêu trước nút | Còn thiếu sau khi trừ | Kết thúc ở lá? |
|---:|---:|---:|---:|---|
| 1 | 5 | 22 | 17 | Không |
| 2 | 4 | 17 | 13 | Không |
| 3 | 11 | 13 | 2 | Không |
| 4 | 2 | 2 | **0** | **Có → True** |

Nếu target=26, đường `5→8→13` cho tổng 26 cũng đúng vì 13 là lá.
Muốn debug tổng, phải viết giá trị **trước khi trừ** bên cạnh giá trị **sau khi trừ**, không chỉ in `True` cuối.

In [ ]:
path_values112=[5,4,11,2]
need112=22
for depth,value in enumerate(path_values112,start=1):
    before=need112;need112-=value
    print('tầng',depth,'node',value,'trước',before,'sau',need112)
assert need112==0

### 7.2. Trace tay nhánh không hợp lệ dù tổng có lúc bằng target

Tạo cây `[1,0,3,5]`, với target=1: sau khi đi qua gốc 1, `remaining` bằng 0,
nhưng gốc có con nên không thể trả True.
Đi vào nút 0 bên trái, remaining vẫn là 0, nhưng nút ấy có con 5,
cũng chưa được phép dừng. Đến lá 5 thì remaining thành -5; đường này không khớp.
Đường bên phải `1→3` có tổng 4; cũng không khớp. Kết luận **False**.

Đây là phản ví dụ để chống lỗi *so tổng ở node giữa mà không xét lá*.
Một test khó mà ngắn thường tốt hơn hàng chục test dài ngẫu nhiên.

In [ ]:
tricky112=tree([1,0,3,5])
print('cây:',show_tree(tricky112))
print('mục tiêu 1:',solve(tricky112,1))
print('mục tiêu 6:',solve(tricky112,6))
assert not solve(tricky112,1)
assert solve(tricky112,6)

In [ ]:
def trace_path(node,rem,level=0):
    if node is None:return False
    print(' '*level,f'node {node.val}, còn {rem}')
    if node.left is None and node.right is None:
        print(' '*level,'lá ->',node.val==rem);return node.val==rem
    return trace_path(node.left,rem-node.val,level+2) or trace_path(node.right,rem-node.val,level+2)
trace_path(tree([5,4,8,11,None,13,4,7,2,None,None,None,1]),22)

### 7.3. Biến thể trace không đệ quy: stack lưu `(node, remaining)`

DFS không nhất thiết phải viết bằng đệ quy. Ta có thể tự giữ stack các trạng thái `(node,remaining)`.
Mỗi lần lấy một trạng thái, nếu node là lá thì so `node.val` với remaining;
ngược lại thêm các con với `remaining - node.val`.

Lưu ý **thứ tự**: vì stack LIFO, muốn duyệt trái trước thường thêm con phải rồi mới thêm con trái.
Đối với câu hỏi True/False, duyệt trái trước hay phải trước không đổi kết quả,
nhưng nó làm thứ tự dòng trace khác nhau. Stack tường minh tránh giới hạn độ sâu lời gọi Python.

In [ ]:
def path_sum_iterative_stack(root,target):
    if root is None:return False
    stack=[(root,target)]
    while stack:
        node,rem=stack.pop()
        print('pop node',node.val,'mục tiêu còn',rem,'stack length',len(stack))
        if node.left is None and node.right is None:
            if node.val==rem:return True
            continue
        new_rem=rem-node.val
        if node.right:stack.append((node.right,new_rem))
        if node.left:stack.append((node.left,new_rem))
    return False
assert path_sum_iterative_stack(tree([1,2,3]),3)
assert not path_sum_iterative_stack(None,0)

### 7.4. Từ stack sang BFS: cùng trạng thái, đổi thứ tự lấy

Nếu thay stack bằng queue, mỗi trạng thái vẫn gồm `(node,remaining)`,
nhưng BFS duyệt cây theo tầng thay vì theo nhánh sâu.
Bài #112 không đòi đường ngắn nhất, nên cả DFS và BFS đều đúng và O(n) thời gian tệ nhất.

**Lựa chọn kỹ thuật theo đề:** cần đường ngắn nhất không trọng số → BFS có lợi thế;
chỉ cần biết có tồn tại đường hay không → DFS đơn giản và ít bộ nhớ trên cây rộng;
muốn tránh recursion sâu → DFS stack hoặc BFS queue. Đừng học máy móc rằng mọi bài Tree đều cần BFS.

In [ ]:
def path_sum_bfs112(root,target):
    if root is None:return False
    q=deque([(root,target)])
    while q:
        node,rem=q.popleft()
        if node.left is None and node.right is None and node.val==rem:
            return True
        rem-=node.val
        if node.left:q.append((node.left,rem))
        if node.right:q.append((node.right,rem))
    return False
for t in (22,26,1,100):
    root=tree([5,4,8,11,None,13,4,7,2,None,None,None,1])
    assert path_sum_bfs112(root,t)==solve(root,t)
print('PASS BFS = DFS trên bốn mục tiêu')

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Bất biến: `remaining` đúng bằng `targetSum - tổng các node trước node hiện tại`. Tại lá, tổng đường đi bằng đích khi và chỉ khi `node.val == remaining`. Đệ quy thử mọi đường gốc-lá hợp lệ; phép OR chỉ kết thúc khi tìm ra một đường đúng.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
from typing import Optional
class Solution:
    def hasPathSum(self, root: Optional[TreeNode], targetSum: int):
        def walk(node,remaining):
            if node is None:return False
            if node.left is None and node.right is None:
                return node.val==remaining
            rest=remaining-node.val
            return walk(node.left,rest) or walk(node.right,rest)
        return walk(root,targetSum)

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
arr=[5,4,8,11,None,13,4,7,2,None,None,None,1]
for data,t,ans in [(arr,22,True),(arr,5,False),([],0,False),([1],1,True),([1,2,3],4,True),([1,-2,-3,1,3,-2,None,-1],-1,True)]:
    assert solve(tree(data),t)==ans
    assert Solution().hasPathSum(tree(data),t)==ans
print('PASS 6 ca tổng đường đi')

### 10.1. Bộ kiểm thử phải có số âm, zero và đường kết thúc không phải lá

Một thuật toán đúng cần xử lý các giá trị âm, số 0 và cây lệch.
- `root=None`, `target=0`: **False** (không có đường).
- `[0]`, `target=0`: **True** (gốc chính là lá).
- `[1,2,3]`, `target=1`: **False** (gốc chưa là lá).
- `[-2,None,-3]`, `target=-5`: **True**.
- `[1,0,2,5]`, `target=1`: **False**, dù ở node giữa có tổng 1.

Trước khi chạy, thử tự tính đường đi gốc→lá của mỗi test; test chống lại những giả định sai
thường dạy bạn nhiều hơn test sao chép đúng ví dụ đề.

In [ ]:
path_cases112=[([],0,False),([0],0,True),([1,2,3],1,False),
               ([-2,None,-3],-5,True),([1,0,2,5],1,False),
               ([1,2,3],4,True),([1,2,3],3,True)]
for values,t,expected in path_cases112:
    root=tree(values)
    assert solve(root,t)==baseline(root,t)==path_sum_bfs112(root,t)==expected
    assert Solution().hasPathSum(root,t)==expected
print('PASS',len(path_cases112),'test cơ bản + edge cases')

### 10.2. Đối chiếu với bộ kiểm tra độc lập bằng cách liệt kê mọi đường gốc→lá

Kiểm chứng mạnh hơn khi thuật toán kiểm tra dùng một ý tưởng **khác**.
Hàm dưới liệt kê mọi đường gốc→lá (mỗi đường là list các số),
rồi trả `any(sum(path)==target for path in paths)`.
Đây là *oracle* kiểm thử: do tạo list đường đi nên tốn bộ nhớ hơn lời giải chính,
nhưng dễ đọc và tự kiểm chứng bằng mắt.

Ta còn sinh cây nhỏ ngẫu nhiên bằng seed cố định, so các câu trả lời
để phát hiện sai sót khó thấy khi chỉ thử các input của đề.

In [ ]:
def all_root_leaf_paths112(node):
    if node is None:return []
    if node.left is None and node.right is None:return [[node.val]]
    paths=[]
    for child in (node.left,node.right):
        for p in all_root_leaf_paths112(child):paths.append([node.val]+p)
    return paths
import random
rng112=random.Random(112)
def make_random112(level):
    if level==0 or rng112.random()<0.26:return None
    return TreeNode(rng112.randint(-6,6),make_random112(level-1),make_random112(level-1))
for _ in range(160):
    root=make_random112(6)
    paths=all_root_leaf_paths112(root)
    for target in (-9,-3,0,3,9):
        oracle=any(sum(p)==target for p in paths)
        assert solve(root,target)==oracle
print('PASS 160 cây × 5 target, đối chiếu liệt kê đường đi')

## 11. Phân tích Time/Space Complexity

Cây có `n` node, độ sâu `h`: **Time O(n)** tệ nhất; **Space O(h)** recursion. Không tạo danh sách chứa mọi đường đi.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Thấy `remaining == 0` ở node **không phải lá** rồi trả True.
- ❌ Quên trừ `node.val` trước khi đi con.
- ❌ Cho cây rỗng và target=0 trả True: không có đường gốc–lá.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Lỗi sai: `if remaining == 0: return True` trước khi đến lá

Phát biểu này nghe hợp lý vì “đã đủ tổng”, nhưng đề yêu cầu **đường đến lá**.
Ví dụ root 5 có con 7, target=5: tổng 5 đúng ở gốc, nhưng gốc chưa phải lá,
và đường hợp lệ duy nhất có tổng 12, nên đáp án #112 là False.

Cần đổi điều kiện thành: tại node *lá*, trả `node.val == remaining`;
ở node giữa luôn đi tiếp để hoàn thành đường. Code sai được tách riêng trong hàm `wrong_...` dưới đây
để bạn thấy ngay lỗi, không đưa đoạn sai vào cell submit.

In [ ]:
def wrong_early_success112(node,remaining):
    if node is None:return False
    remaining-=node.val
    if remaining==0:return True  # SAI: có thể chưa phải lá
    return wrong_early_success112(node.left,remaining) or wrong_early_success112(node.right,remaining)
root_wrong112=tree([5,7])
print('sai:',wrong_early_success112(root_wrong112,5),'đúng:',solve(root_wrong112,5))
assert wrong_early_success112(root_wrong112,5) is True
assert solve(root_wrong112,5) is False

### 12.2. Lỗi sai: tỉa nhánh khi `remaining < 0`

Vì `target` có thể âm và giá trị node có thể âm, việc remaining âm **không đồng nghĩa bất khả thi**.
Ví dụ target=3 và đường `[5,-2]`: sau nút 5, remaining=3-5=-2;
nhưng lá -2 làm tổng trở thành 3, nên đáp án True.

Vì thế không được thêm `if remaining<0: return False` vào lời giải tổng quát.
Chỉ khi đề bảo đảm mọi giá trị node không âm, ta mới có thể cân nhắc tỉa như vậy
(và vẫn phải kiểm tra ranh giới/lá cẩn thận). Đọc **constraints** trước khi đề xuất tối ưu.

In [ ]:
negative_path112=tree([5,-2])
remaining_after_root=3-negative_path112.val
print('còn thiếu sau root:',remaining_after_root)
print('lá -2 có khớp?',negative_path112.left.val==remaining_after_root)
print('đáp án đúng:',solve(negative_path112,3))
assert remaining_after_root==-2 and solve(negative_path112,3)

## 13. Nhận diện dạng bài và vận dụng

"Có tồn tại đường đi ... tổng bằng k?" trong cây → DFS và trạng thái **phần còn lại**, base case chỉ tại node lá.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Đổi `targetSum` từ 22 thành 26 trên cùng cây và tự tìm đường nào cho tổng bằng 26 (nếu có).

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
root=tree([5,4,8,11,None,13,4,7,2,None,None,None,1]);print('22:',solve(root,22),'26:',solve(root,26))

### 14.1. Pattern “trạng thái còn thiếu” dùng lại ở đâu?

Cùng một công thức `remaining - current`, nhưng cách kết thúc phụ thuộc yêu cầu:

| Bài toán | Điều kiện kết thúc | Cách gộp kết quả con |
|---|---|---|
| #112: Có đường? | Tại lá và vừa đủ tổng | `left or right` |
| Đếm đường gốc→lá | Tại lá vừa đủ tổng trả 1 | `left + right` |
| Lấy một đường | Tại lá trả một list | ưu tiên nhánh có list khác rỗng |
| Lấy tất cả đường | Tại lá trả list chứa một path | nối các danh sách |

Chuyển giao không phải chép mỗi `target -= root.val` mà là hiểu **ý nghĩa trạng thái và kiểu trả về**.
Thử một cây nhỏ có **hai** đường cùng tổng để xem True/False và số đếm khác nhau.

In [ ]:
root_multi112=tree([1,2,2])
print('bài #112 có đường tổng 3?',solve(root_multi112,3))
print('có bao nhiêu đường tổng 3?',answer_count_path_sum(root_multi112,3) if 'answer_count_path_sum' in globals() else 'sẽ trả lời ở phần Đáp án')
assert solve(root_multi112,3)

### 14.2. Bài tập nhận diện: chọn DFS hay BFS?

Nếu đề chỉ hỏi **tồn tại** thì DFS `or` đủ; nếu hỏi **đường có ít cạnh nhất**
trong đồ thị không trọng số thì BFS đáng ưu tiên; nếu hỏi **mọi đường**
thì cần lưu và tích lũy kết quả, không thể return tại đáp án đầu tiên.

Tương tự, kỹ thuật **remaining target** có thể chuyển sang chia tiền, chọn số có tổng mong muốn,
nhưng phải phân biệt giới hạn “một nút/ phần tử được chọn mấy lần”.
Bài #112 đường đi bị ràng buộc bởi liên kết cha-con thật, không phải chọn tập con tùy ý.
Đoạn code ngắn này liệt kê đường thật để thấy khác biệt.

In [ ]:
roots=[tree([1,2,3]),tree([1,None,2,None,3]),None]
for root in roots:
    print('cây:',show_tree(root),'-> các đường:',all_root_leaf_paths112(root))
assert all_root_leaf_paths112(tree([1,2,3]))==[[1,2],[1,3]]

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Với `target=22` và `path=[5,4,11,2]`, dự đoán giá trị `remaining` sau khi trừ từng phần tử từ trái qua phải.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [ ]:
remaining=22
for value in [5,4,11,2]:
    remaining-=value
    print(value,"-> còn",remaining)

### Bài 2 — Trace bằng tay (hiểu bản chất)

Với cây `[5,4,8,11,None,13,4,7,2,None,None,None,1]` và target `22`, chỉ ra một đường đi **gốc→lá** thỏa tổng; giải thích vì sao đường dừng giữa chừng tại node 11 chưa hợp lệ.

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_has_path_sum(root,target)` kiểm tra sự tồn tại đường gốc→lá có tổng bằng target.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [ ]:
def exercise_has_path_sum(root,target):
    # TODO: root None -> False; khi là lá so với target.
    # TODO: trừ root.val trước khi sang con.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Biến thể: `exercise_one_path_sum(root,target)` trả **một danh sách giá trị** trên đường gốc→lá hợp lệ; không có thì `[]`.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [ ]:
def exercise_one_path_sum(root,target):
    # TODO: dùng backtracking hoặc đệ quy trả list; chọn trái trước.
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_count_path_sum(root,target)` đếm **tất cả** đường gốc→lá có tổng bằng target, không chỉ trả True/False.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [ ]:
def exercise_count_path_sum(root,target):
    # TODO: tới lá trả 1 nếu đúng tổng; tổng hợp số đường ở trái + phải.
    raise NotImplementedError

### Bài 6 — TODO: liệt kê **mọi** đường gốc→lá có tổng bằng target

Hãy giải `exercise_all_path_sums(root,target)` trả **danh sách các đường** (mỗi đường là list giá trị).
Nếu không có đường hợp lệ trả `[]`; nếu `root=None` cũng trả `[]`.
Ví dụ cây `[1,2,2]`, target 3 có hai đường trùng giá trị nên kết quả là `[[1,2],[1,2]]`:
chúng là **hai đường khác nhau** về nút, không được gộp thành một.

Gợi ý 1: ở lá thành công trả `[[node.val]]` (một danh sách chứa một đường).
Gợi ý 2: gọi đệ quy cho trái/phải với `target-node.val`, lấy các đường con và thêm `node.val` vào đầu.
Gợi ý 3: kết quả là `left_paths + right_paths`, không dùng `or` nếu muốn giữ mọi đường.

In [ ]:
def exercise_all_path_sums(root,target):
    # TODO: trả list[list[int]] gồm mọi đường root->leaf có tổng target.
    # Không gọi sẵn oracle all_root_leaf_paths112, hãy tự luyện recursion.
    raise NotImplementedError

# Tự viết rồi mới chạy:
# assert exercise_all_path_sums(tree([1,2,2]),3)==[[1,2],[1,2]]

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Đường đi hợp lệ bắt đầu ở **gốc** và kết thúc tại **lá**, không được dừng ở node giữa.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Mỗi lời gọi truyền `target - node.val`; tìm đường dùng `or`, đếm đường dùng `+`.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Tại lá, so `node.val == target`. Còn lại trả kết quả cho con trái / con phải với target đã trừ.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Các dòng: `5 -> còn 17`, `4 -> còn 13`, `11 -> còn 2`, `2 -> còn 0`. Chỉ kết luận đúng khi **đã tới lá**.

### Bài 2 — Trace tay

Đường đi `5→4→11→2` có tổng `22`. Node `11` vẫn có hai con, nên dừng tại đây sẽ sai dù một tổng tạm thời nào đó khớp target.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [ ]:
def answer_has_path_sum(root,target):
    if root is None: return False
    if root.left is None and root.right is None: return root.val==target
    remaining=target-root.val
    return answer_has_path_sum(root.left,remaining) or answer_has_path_sum(root.right,remaining)

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [ ]:
def answer_one_path_sum(root,target):
    if root is None: return []
    if root.left is None and root.right is None:
        return [root.val] if root.val==target else []
    left=answer_one_path_sum(root.left,target-root.val)
    if left: return [root.val]+left
    right=answer_one_path_sum(root.right,target-root.val)
    return [root.val]+right if right else []

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [ ]:
def answer_count_path_sum(root,target):
    if root is None: return 0
    if root.left is None and root.right is None: return int(root.val==target)
    remaining=target-root.val
    return answer_count_path_sum(root.left,remaining)+answer_count_path_sum(root.right,remaining)

### Đáp án bài 6 — kiểu trả về list[list[int]] giúp gộp tất cả nhánh

Ở `None` không có đường nên trả `[]`. Ở lá khớp target, có **một** đường gồm một số,
nên trả `[[node.val]]`; không khớp trả `[]`.
Ở node giữa, gọi cả trái lẫn phải; với từng đường con phù hợp, thêm giá trị node hiện tại vào đầu.
Vì dùng phép nối danh sách, nếu hai đường có giá trị trùng hệt nhau thì vẫn xuất hiện hai lần.

Đây cũng là tiền đề cho LeetCode #113 Path Sum II (Medium).
Do tạo đường đi tường minh, bộ nhớ có thể lớn hơn #112; không dùng công thức O(h) của bài chỉ trả bool.

In [ ]:
def answer_all_path_sums(root,target):
    if root is None:return []
    if root.left is None and root.right is None:
        return [[root.val]] if root.val==target else []
    left=answer_all_path_sums(root.left,target-root.val)
    right=answer_all_path_sums(root.right,target-root.val)
    return [[root.val]+p for p in left+right]
assert answer_all_path_sums(tree([1,2,2]),3)==[[1,2],[1,2]]
assert answer_all_path_sums(tree([1,2,3]),3)==[[1,2]]
assert answer_all_path_sums(None,0)==[]
assert answer_all_path_sums(tree([1,2,3]),99)==[]
print('PASS bài 6: liệt kê tất cả đường phù hợp')

### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [ ]:
r=tree([5,4,8,11,None,13,4,7,2,None,None,None,1])
assert answer_has_path_sum(r,22)
assert not answer_has_path_sum(r,100)
assert answer_one_path_sum(r,22)==[5,4,11,2]
assert answer_one_path_sum(r,100)==[]
assert answer_count_path_sum(tree([1,1,1]),2)==2
assert answer_count_path_sum(None,0)==0
print("PASS: đáp án 3 bài coding của phần luyện tập")

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Thay vì mang `total` đã qua, mang `remaining` còn thiếu; đi vào node giá trị `x` thì cập nhật `remaining-x`. Ở lá, chỉ cần kiểm tra `remaining == node.val`.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.